# 08 — Label Propagation

No fine-tuning loop: embed the 5%-per-class labeled seed and the full
unlabeled pool with MiniLM (on raw text), build a k-NN graph over all of
them, and propagate the seed labels through the graph via
`sklearn.semi_supervised.LabelSpreading`. Full unlabeled pool, no sampling
cap (`config.SAMPLE_SIZE`).

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import pandas as pd

from utils import config
from utils.data import stratified_sample
from utils.embeddings import get_sentence_embeddings
from utils.label_propagation import run_label_propagation
from utils.metrics import evaluate_label_quality
from utils.samples import save_full_output, save_label_samples

In [2]:
labeled_df = pd.read_parquet(config.PROCESSED_DIR / "labeled.parquet")
unlabeled_df = pd.read_parquet(config.PROCESSED_DIR / "unlabeled.parquet")
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")

unlabeled_sample = stratified_sample(unlabeled_df, config.SAMPLE_SIZE, seed=config.SEED, label_col="true_label")

overlap = set(unlabeled_sample["text"]) & set(test_clean["text"])
assert len(overlap) == 0, f"{len(overlap)} rows leaked between train pool and test set"
print(f"Labeled seed: {len(labeled_df)} | Unlabeled pool: {len(unlabeled_sample)} | Test: {len(test_clean)}")

Labeled seed: 16 | Unlabeled pool: 303 | Test: 80


In [3]:
cache_name = f"minilm_labelprop_L{len(labeled_df)}_U{len(unlabeled_sample)}"
combined_texts = labeled_df["text"].tolist() + unlabeled_sample["text"].tolist()
combined_embeddings = get_sentence_embeddings(combined_texts, cache_name)

labeled_embeddings = combined_embeddings[:len(labeled_df)]
unlabeled_embeddings = combined_embeddings[len(labeled_df):]
assert unlabeled_embeddings.shape[0] == len(unlabeled_sample)
print(f"Embedded {len(combined_texts)} texts (MiniLM, {combined_embeddings.shape[1]}-d)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embedded 319 texts (MiniLM, 384-d)


In [4]:
predicted_labels, confidence = run_label_propagation(
    labeled_embeddings, labeled_df["label"].to_numpy(), unlabeled_embeddings,
    kernel="knn", n_neighbors=7)

label_quality = evaluate_label_quality(
    true_labels=unlabeled_sample["true_label"].to_numpy(),
    pseudo_labels=predicted_labels,
    confidence_scores=confidence)
print("Label propagation quality:", label_quality)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_label_propagation.json", "w") as f:
    json.dump(label_quality, f, indent=2)
print("Saved label propagation results.")

Label propagation quality: {'Label Accuracy': 0.35313531353135313, 'Label Macro F1': 0.33616060446452756, 'Coverage': np.float64(1.0), 'Mean Confidence': 0.7491720078067847, 'Median Confidence': 0.8227407050186295}
Saved label propagation results.


In [5]:
save_label_samples(
    unlabeled_sample["text"], predicted_labels, unlabeled_sample["true_label"].to_numpy(),
    config.CLASS_NAMES, confidence=confidence, n_per_class=2, seed=config.SEED,
    path=config.RESULTS_DIR / "sample_labels_label_propagation.csv")
save_full_output(
    unlabeled_sample["text"], predicted_labels, unlabeled_sample["true_label"].to_numpy(),
    config.CLASS_NAMES, confidence=confidence,
    extra_columns={"summary": unlabeled_sample["summary"].tolist()},
    path=config.RESULTS_DIR / "full_labels_label_propagation.csv")
print("Saved sample + full-row outputs for label_propagation.")

Saved sample + full-row outputs for label_propagation.
